# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [ ]:
#load all necessary libraries
import re 
import Bio.SeqIO
import Bio.Seq

#one important note - i am unable to scilence the ever-persistent AI chat bot so a few things here may have been unintentionally been AI.
#but I cited my friend and AI to the best of my ability 


In [53]:
#compile list from fastq and parse length of each read
barcodes_fastq = list(Bio.SeqIO.parse("barcodes_R1.fastq", format = "fastq"))
print(len(barcodes_fastq))

#check contents of each read
read1 = barcodes_fastq[0]
print("id:      ", read1.id)
print("sequence:", read1.seq)
print("length:  ", len(read1.seq))
print("quality: ", read1.letter_annotations["phred_quality"])


10000
id:       HISEQ:621:HMJGNBCX2:1:1101:1797:2150
sequence: GCTTAAGTTATTTAGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTGAAAGGGAGTTCTGCTCCATCAGGCCAAGATCGGAAGAGCACACGTCTGAACTCCAGTCACATGTCAGAATCTCGTATGCCGTCTTCTGCTTGAAAAAAAAAAAAAAGCTGAAATTAATAATTTTGAAACCAGTTTTGTAAACGCAGCACTAAAATGAAGGCATGCAACGACGATGTTTATTGACACGGAATAGCAGA
length:   250
quality:  [38, 38, 38, 38, 38, 38, 40, 40, 40, 38, 40, 40, 40, 40, 38, 40, 40, 40, 40, 38, 40, 40, 40, 40, 40, 40, 40, 38, 38, 40, 38, 40, 40, 40, 40, 40, 40, 40, 40, 40, 38, 40, 40, 40, 40, 38, 38, 40, 40, 40, 38, 40, 38, 40, 40, 40, 38, 38, 40, 40, 40, 40, 38, 40, 40, 40, 40, 38, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 38, 40, 40, 40, 40, 40, 40, 40, 40, 38, 40, 40, 40, 40, 40, 38, 38, 40, 40, 40, 40, 38, 38, 38, 38, 40, 38, 32, 38, 38, 40, 38, 38, 38, 38, 38, 40, 38, 40, 40, 40, 27, 32, 38, 38, 38, 40, 40, 38, 38, 40, 40, 38, 38, 40, 38, 40, 38, 38, 32, 38, 40, 38, 38, 40, 38, 38, 38, 38, 40, 40, 40, 40, 38, 32, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2

In [47]:
# isolate sequence for each read and return the reverse complement sequence 

for read in barcodes_fastq:
    seq = read.seq
    rev_comp_seq = seq.reverse_complement()
    print(rev_comp_seq)


...


In [ ]:
#set up a function to read the barcode from the reverse complement sequence, given a known upstream sequence and barcode length

def protein_id(seqread, bclen=16, upstream="AGGCGGCCGC"):
    
    """Classify a read as HA, NA, or None.
    """
    ha_id = "CCGGATTTGCATATAATGATGCACCAT"
    na_id = "CACGATAGATAAATAATAGTGCACCAT"

    seqread_upper=str(seqread).upper()

    seqread_original=seqread_upper = str(Bio.Seq.Seq(seqread_upper).reverse_complement())

    if ha_id in seqread_original:
        return "HA"
    elif na_id in seqread_original:
        return "NA"
    else:
        return None



In [70]:
# This is a helper function from class notes: it runs a parser function against a list of test cases and reports the results
def run_tests(parser_function, test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return f"{n_failed} tests failed"


#test cases from Dunham (classmate)

# Create a known HA sequence and put it in sequencing-read orientation
ha_sequence = (
    "CCGGATTTGCATATAATGATGCACCAT"
    + "AGGCGGCCGC"
    + "ACGTACGTACGTACGT"
)

ha_test_read = Bio.Seq.Seq(ha_sequence).reverse_complement()


# Create a known NA sequence and put it in sequencing-read orientation
na_sequence = (
    "CACGATAGATAAATAATAGTGCACCAT"
    + "AGGCGGCCGC"
    + "ACGTACGTACGTACGT"
)

na_test_read = Bio.Seq.Seq(na_sequence).reverse_complement()

test_cases = [(ha_test_read, "HA", "HA test case"),
    (na_test_read, "NA", "NA test case"),
    ("ACGTACGTACGTACGT", None, "No barcode test case"),
]

In [ ]:
#run test cases, hope all say "PASS"
run_tests(protein_id, test_cases)

PASS  HA test case
PASS  NA test case
PASS  No barcode test case


'0 tests failed'

In [ ]:
#here I use the protein_id function to find how many times NA and HA appear in the barcodes_R1.fastq file

barcodes_fastq = list(Bio.SeqIO.parse("barcodes_R1.fastq", format = "fastq"))

#start counters at zero to add to through for loop
ha_count = 0
na_count = 0 
other = 0

for sequence in barcodes_fastq:
    id = protein_id(sequence.seq)
    if id == "HA":
        ha_count += 1
    elif id == "NA":
        na_count += 1
    else:
        other +=1

print("total reads = ", len(barcodes_fastq))
print("HA reads = ", ha_count)
print("NA reads = ", na_count)
print("Other reads = ", other)

total reads =  10000
HA reads =  5409
NA reads =  4122
Other reads =  469


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [91]:
# First is to create a function to parse the sequences for a barcode

def read_barcode(seqread, bclen=16, upstream="AGGCGGCCGC"):
    """Identify a barcode with a known upstream sequence.

    Args:
        seqread: Nucleotide sequence read (a Seq object or a string) matching
            in reverse orientation.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode (forward orientation).

    Returns:
        The barcode in the forward orientation, as an upper-case string, or
        None if the read does not match the expected design.

    Example:
        >>> read_barcode(Bio.Seq.Seq("TTTTTTTTTTTTTTTTGCGGCCGCCT"), bclen=16)
        'AAAAAAAAAAAAAAAA'
    """
    # define the sequence and barcode pattern, based on class notes and advice from classmates 
    seqread_upper=str(seqread).upper()
    seqread_original=seqread_upper = str(Bio.Seq.Seq(seqread_upper).reverse_complement())
    #Dunham helped explain here the f strings and why so many brackets are necessary
    barcode_pattern=re.compile(f"{upstream}(?P<barcode>[ACTGN]{{{bclen}}})")

    #this line is saying if the barcode is found in the original sequence, it is a match. if it is a match, add it to the group "barcode"
    match = barcode_pattern.search(seqread_original)

    if match:
        return match.group("barcode")
    else:
        return None


In [92]:
#check function for function (does it work with a few test cases)

barcode_1 = "TCAGCGATGAATGTGGGCGGCCGCCT" #should pass, barcode = "CCACATTCATCGCTGA" 

barcode_2 = "TCAGCGATGAGCGGCCGCCT" #should fail, barcode = None

barcode_3 = "TTTTTTTTTTTTTTTTGCGGCCGCCT" #should pass

#use run tests, generate a test cases folder for the three tests above and the expected outcomes
test_cases_bc = [(barcode_1, "CCACATTCATCGCTGA", "Valid barcode in forward direction"),
    (barcode_2, None, "barcode too short"),
    (barcode_3, "AAAAAAAAAAAAAAAA", "valid barcode in forward direction"),]

run_tests(read_barcode, test_cases_bc)

PASS  Valid barcode in forward direction
PASS  barcode too short
PASS  valid barcode in forward direction


'0 tests failed'

In [94]:
#use a for loop to iterate through sequences in barcodes_R1.fastq and sort all barcodes into respective dictionaries

#make empty dictionaries for HA and NA
ha_bc_dict = {}
na_bc_dict = {}

#make counter for all invalid barcodes not counted in dictionaries
ha_invalid = 0
na_invalid = 0

for reads in barcodes_fastq:
    id = protein_id(reads.seq)
    bc = read_barcode(reads.seq)
    if id == "HA":
            # assigns that if bc is invalid, then invalid count increases, otherwise move to else...
        if bc is None:
            ha_invalid +=1
            #if not invalid, then determine if the barcode is already in the dictionary. if it is, add to the number of counts. 
            #   if it isnt, add to the dictionary with count of 1
        else:
            if bc in ha_bc_dict:
                ha_bc_dict[bc] += 1
            else:
                ha_bc_dict[bc] = 1
    elif id == "NA":
        if bc is None:
            na_invalid +=1
        else:
            if bc in na_bc_dict:
                na_bc_dict[bc] += 1
            else:
                na_bc_dict[bc] = 1

print("The number of unique HA barcodes is ", len(ha_bc_dict))
print("The number of unique NA barcodes is ", len(na_bc_dict))
print("\n")
print("There are", ha_invalid, "invalid HA barcodes", "and", na_invalid, "invalid NA barcodes in the sequencing run.")

The number of unique HA barcodes is  283
The number of unique NA barcodes is  140


There are 159 invalid HA barcodes and 210 invalid NA barcodes in the sequencing run.


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [ ]:
#search through the dictionary to identify the key with the higest number for HA and NA. 
#I had some help from AI to find the "max" function used here

for bc in ha_bc_dict:
    if ha_bc_dict[bc] == max(ha_bc_dict.values()):
        print("The most common HA barcode is", bc, "with a count of", ha_bc_dict[bc])

for bc in na_bc_dict:   
    if na_bc_dict[bc] == max(na_bc_dict.values()):
        print("The most common NA barcode is", bc, "with a count of", na_bc_dict[bc])

The most common HA barcode is CCCGACCCGACATTAA with a count of 155
The most common NA barcode is ACCAGTTCTCCCCGGG with a count of 152
